# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
spaceship['deck'] = spaceship['Cabin'].str.split('/').str[0]
spaceship['side'] = spaceship['Cabin'].str.split('/').str[2]
spaceship['total_expenses'] = spaceship[['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']].sum(axis=1)
spaceship['no_expenses'] = (spaceship['total_expenses'] == 0).astype(int)
spaceship = spaceship.drop(columns=['PassengerId', 'Cabin', 'Name'])
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

num_features = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'total_expenses']
cat_features = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'deck', 'side']

num_transformer = Pipeline([('imputer', SimpleImputer(strategy='median')),('scaler', StandardScaler())])

cat_transformer = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))])

preprocessor = ColumnTransformer([('num', num_transformer, num_features),('cat', cat_transformer, cat_features)], remainder='drop')

**Perform Train Test Split**

In [4]:
from sklearn.model_selection import train_test_split
X = spaceship.drop(columns='Transported')
y = spaceship['Transported']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [5]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

bagging_pipeline = Pipeline([('preprocessor', preprocessor),('classifier', BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42), n_estimators=100, bootstrap=True, random_state=42, n_jobs=-1))])
bagging_pipeline.fit(X_train, y_train)
y_pred_bag = bagging_pipeline.predict(X_test)
print("Bagging Accuracy:", accuracy_score(y_test, y_pred_bag))
print(classification_report(y_test, y_pred_bag))

pasting_pipeline = Pipeline([('preprocessor', preprocessor),('classifier', BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42), n_estimators=100, bootstrap=False, random_state=42, n_jobs=-1))])
pasting_pipeline.fit(X_train, y_train)
y_pred_paste = pasting_pipeline.predict(X_test)
print("Pasting Accuracy:", accuracy_score(y_test, y_pred_paste))

Bagging Accuracy: 0.8027602070155262
              precision    recall  f1-score   support

       False       0.78      0.84      0.81       863
        True       0.83      0.77      0.80       876

    accuracy                           0.80      1739
   macro avg       0.80      0.80      0.80      1739
weighted avg       0.80      0.80      0.80      1739

Pasting Accuracy: 0.7464059804485337


- Random Forests

In [6]:
from sklearn.ensemble import RandomForestClassifier
rf_pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1))])
rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Random Forest Accuracy: 0.8004600345025877
              precision    recall  f1-score   support

       False       0.78      0.84      0.81       863
        True       0.83      0.77      0.79       876

    accuracy                           0.80      1739
   macro avg       0.80      0.80      0.80      1739
weighted avg       0.80      0.80      0.80      1739



- Gradient Boosting

In [7]:
from sklearn.ensemble import HistGradientBoostingClassifier
gb_pipeline = Pipeline([('preprocessor', preprocessor),('classifier', HistGradientBoostingClassifier(random_state=42, max_iter=100))])
gb_pipeline.fit(X_train, y_train)
y_pred_gb = gb_pipeline.predict(X_test)
print("Gradient Boosting Accuracy:", accuracy_score(y_test, y_pred_gb))
print(classification_report(y_test, y_pred_gb))

Gradient Boosting Accuracy: 0.8142610695802185
              precision    recall  f1-score   support

       False       0.83      0.79      0.81       863
        True       0.80      0.84      0.82       876

    accuracy                           0.81      1739
   macro avg       0.81      0.81      0.81      1739
weighted avg       0.81      0.81      0.81      1739



- Adaptive Boosting

In [8]:
from sklearn.ensemble import AdaBoostClassifier
ada_pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', AdaBoostClassifier(n_estimators=100, random_state=42))])
ada_pipeline.fit(X_train, y_train)
y_pred_ada = ada_pipeline.predict(X_test)
print("AdaBoost Accuracy:", accuracy_score(y_test, y_pred_ada))
print(classification_report(y_test, y_pred_ada))

AdaBoost Accuracy: 0.7832087406555491
              precision    recall  f1-score   support

       False       0.79      0.77      0.78       863
        True       0.78      0.80      0.79       876

    accuracy                           0.78      1739
   macro avg       0.78      0.78      0.78      1739
weighted avg       0.78      0.78      0.78      1739



Which model is the best and why?

In [9]:
#HistGradientBoostingClassifier is the best model because it natively manages NAN, correctiong mistakes too like other boosting methods.